# 🎨 TRELLIS: Image to 3D Converter (WORKING VERSION)

Convert images to 3D models using Microsoft's TRELLIS!

## Setup

1. **Enable GPU**: Runtime → Change runtime type → T4 GPU
2. **Run all cells in order**

---

## ✅ Step 1: Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 📦 Step 2: Install Dependencies (COMPLETE)

In [ ]:
import sys
from pathlib import Path

print("📦 Installing TRELLIS dependencies...\n")

# Install all required packages
print("[1/6] Core packages...")
!pip install -q transformers diffusers accelerate

print("[2/6] Image processing...")
!pip install -q pillow opencv-python imageio
!pip install -q rembg[gpu]  # This was missing!

print("[3/6] 3D packages...")
!pip install -q trimesh[easy] pygltflib
!pip install -q kaolin -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html || echo "Kaolin optional"

print("[4/6] ML utilities...")
!pip install -q einops omegaconf timm torchvision
!pip install -q huggingface_hub

print("[5/6] Additional dependencies...")
!pip install -q scipy scikit-image matplotlib
!pip install -q spconv-cu120 || echo "spconv optional"

# Clone TRELLIS
print("[6/6] Downloading TRELLIS...")
TRELLIS_DIR = Path("/content/TRELLIS")
if not TRELLIS_DIR.exists():
    !git clone -q https://github.com/microsoft/TRELLIS.git /content/TRELLIS
    !cd /content/TRELLIS && git submodule update --init --recursive --quiet

sys.path.insert(0, str(TRELLIS_DIR))

print("\n✅ Installation complete!")

## 🔍 Step 3: Load TRELLIS

In [ ]:
print("Loading TRELLIS...\n")

import sys
sys.path.insert(0, '/content/TRELLIS')

from trellis.pipelines import TrellisImageTo3DPipeline
from PIL import Image
import torch
import numpy as np

print("✅ TRELLIS imported successfully!\n")

# Load model
MODEL_NAME = "JeffreyXiang/TRELLIS-image-large"
print(f"Loading model: {MODEL_NAME}")
print("Downloading ~5GB (first time only)...\n")

pipeline = TrellisImageTo3DPipeline.from_pretrained(MODEL_NAME)
pipeline = pipeline.to("cuda" if torch.cuda.is_available() else "cpu")

print("\n✅ Model loaded!")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 🛠️ Step 4: Helper Function

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
from pathlib import Path

def convert_to_3d(image_path, output_path="output.glb", seed=1):
    print(f"\n{'='*70}")
    print(f"Converting: {Path(image_path).name}")
    print(f"{'='*70}\n")
    
    # Load image
    image = Image.open(image_path).convert("RGB")
    print(f"Image: {image.size[0]}x{image.size[1]} pixels")
    
    # Show preview
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis('off')
    plt.title(Path(image_path).name)
    plt.tight_layout()
    plt.show()
    
    # Set seed
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    # Run TRELLIS
    print(f"\n🔮 Running TRELLIS (seed={seed})...")
    print("⏱️ Takes ~30-60 seconds...\n")
    
    outputs = pipeline.run(image, seed=seed)
    print("✓ Inference complete!\n")
    
    # Export
    print("💾 Exporting GLB...")
    
    # TRELLIS outputs: gaussian, radiance_field, mesh
    # Try different export methods
    try:
        # Method 1: Direct export
        if hasattr(outputs, 'export_glb'):
            outputs.export_glb(output_path)
        # Method 2: Mesh export
        elif hasattr(outputs, 'mesh'):
            outputs.mesh.export(output_path)
        # Method 3: Get mesh from dict
        elif isinstance(outputs, dict) and 'mesh' in outputs:
            outputs['mesh'].export(output_path)
        else:
            print(f"⚠️ Unknown output format: {type(outputs)}")
            return None
        
        if Path(output_path).exists():
            size = Path(output_path).stat().st_size / 1e6
            print(f"\n✅ Success!")
            print(f"File: {output_path}")
            print(f"Size: {size:.2f} MB")
            return output_path
        else:
            print("❌ Export failed")
            return None
            
    except Exception as e:
        print(f"❌ Export error: {e}")
        import traceback
        traceback.print_exc()
        return None

print("✅ Ready to convert!")

## 📤 Step 5: Upload Image

In [ ]:
from google.colab import files
import shutil

input_dir = Path("/content/inputs")
input_dir.mkdir(exist_ok=True)

print("📤 Upload your image(s):\n")
uploaded = files.upload()

if uploaded:
    print("\n✅ Uploaded:")
    for name in uploaded:
        dst = input_dir / name
        Path(name).rename(dst)
        print(f"   • {name}")

## 🎨 Step 6: Convert to 3D!

In [ ]:
# Find uploaded images
images = list(Path("/content/inputs").glob("*"))
images = [img for img in images if img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]

if not images:
    print("❌ No images. Upload in Step 5.")
else:
    print(f"Found {len(images)} image(s)\n")
    
    # Convert first image
    result = convert_to_3d(str(images[0]), "/content/output.glb", seed=1)
    
    if result:
        print(f"\n{'='*70}")
        print("📥 DOWNLOAD YOUR 3D MODEL:")
        print(f"{'='*70}\n")
        display(FileLink(result))
        print("\n💡 View at: https://gltf-viewer.donmccurdy.com")

## 🔄 Step 7: Batch Convert (Optional)

In [ ]:
output_dir = Path("/content/outputs")
output_dir.mkdir(exist_ok=True)

images = list(Path("/content/inputs").glob("*"))
images = [img for img in images if img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]

if images:
    print(f"Converting {len(images)} images...\n")
    
    for i, img in enumerate(images, 1):
        print(f"\n[{i}/{len(images)}]")
        out = output_dir / f"{img.stem}.glb"
        convert_to_3d(str(img), str(out), seed=1)
    
    print(f"\n{'='*70}")
    print("📥 Download models:")
    print(f"{'='*70}\n")
    
    for glb in sorted(output_dir.glob("*.glb")):
        size = glb.stat().st_size / 1e6
        print(f"   {glb.name} ({size:.2f} MB)")
        display(FileLink(str(glb)))

## 📦 Step 8: Download All as ZIP

In [ ]:
import shutil
from google.colab import files

output_dir = Path("/content/outputs")
glbs = list(output_dir.glob("*.glb"))

if glbs:
    print(f"Creating zip with {len(glbs)} models...")
    archive = shutil.make_archive("/content/models", 'zip', output_dir)
    print("\n📥 Downloading...")
    files.download(archive)
    print("✅ Done!")
else:
    print("❌ No files to zip. Run Step 7 first.")

---

## 💡 Tips

### View Your Models
- **[gltf-viewer.donmccurdy.com](https://gltf-viewer.donmccurdy.com)** ← Drag & drop!
- [babylon.js/sandbox](https://sandbox.babylonjs.com)
- Import in Blender, Unity, Unreal

### Best Images
✅ Clear, well-lit objects  
✅ Single subject  
✅ Simple background  
✅ 1024x1024+ resolution  

### Performance
- T4 GPU: ~30-60s per image
- Try different seeds (1-100) for variations

### Troubleshooting
- **Out of Memory**: Restart runtime, try one image
- **Session timeout**: Free tier has ~2-3 hour limit
- **Downloads**: Save GLB files before closing

---

## 🎉 Enjoy!

Share your 3D models on:
- [Sketchfab](https://sketchfab.com)
- Social media
- Your projects!

**Happy creating!** ✨